In [ ]:
from google.colab import drive
drive.mount('/content/drive')
exec(open("/content/drive/MyDrive/imdb_peft_project/code/lora-imdb-classifier/00_colab_setup.py").read())

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ /content/drive/MyDrive/imdb_peft_project
✓ /content/drive/MyDrive/imdb_peft_project/checkpoints/roberta_lora
✓ /content/drive/MyDrive/imdb_peft_project/checkpoints/deberta_lora
✓ /content/drive/MyDrive/imdb_peft_project/oof_predictions
✓ /content/drive/MyDrive/imdb_peft_project/results
✓ /content/drive/MyDrive/imdb_peft_project/notebooks
✓ /content/drive/MyDrive/imdb_peft_project/code

Folder structure ready.
Enter GitHub Token: ··········
Repository exists, pulling latest changes...
✓ Pull complete.

Repository path: /content/drive/MyDrive/imdb_peft_project/code/lora-imdb-classifier
SETUP COMPLETE
Drive folder  : /content/drive/MyDrive/imdb_peft_project
GitHub repo   : /content/drive/MyDrive/imdb_peft_project/code/lora-imdb-classifier

Available functions:
  push_to_github('message')        → push code to GitHub
  save_code_to_rep

In [ ]:
!pip install transformers peft accelerate torchao --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 162.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 121.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
import time
import os
from transformers import (RobertaTokenizer, RobertaForSequenceClassification,
                          TrainingArguments, Trainer)
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_score, recall_score, roc_auc_score)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [ ]:
# Ablation results folder
DIRS["ablation"] = f"{DIRS['root']}/ablation_results"
os.makedirs(DIRS["ablation"], exist_ok=True)
print(f"✓ {DIRS['ablation']}")

# Load data
train_df = pd.read_parquet(f"{DIRS['root']}/train_df_v2.parquet")
test_df  = pd.read_parquet(f"{DIRS['root']}/test_df_v2.parquet")
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

✓ /content/drive/MyDrive/imdb_peft_project/ablation_results
Train: 25000 | Test: 25000


In [ ]:
def head_tail_truncate_v2(text, tokenizer, max_len=512, head_len=256):
    """V2: Equal split — first 256 + last 256 tokens."""
    tail_len = max_len - head_len
    tokens = tokenizer(text, add_special_tokens=False,
                       truncation=False, return_tensors=None)
    input_ids      = tokens["input_ids"]
    attention_mask = tokens["attention_mask"]
    if len(input_ids) > max_len - 2:
        input_ids      = input_ids[:head_len] + input_ids[-tail_len:]
        attention_mask = attention_mask[:head_len] + attention_mask[-tail_len:]
    return tokenizer(
        tokenizer.decode(input_ids),
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors=None
    )

class IMDBDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512, head_len=256):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.head_len  = head_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text  = self.df.loc[idx, "text_clean"]
        label = self.df.loc[idx, "label"]
        encoding = head_tail_truncate_v2(
            text, self.tokenizer, self.max_len, self.head_len
        )
        return {
            "input_ids":      torch.tensor(encoding["input_ids"],      dtype=torch.long),
            "attention_mask": torch.tensor(encoding["attention_mask"], dtype=torch.long),
            "labels":         torch.tensor(label,                      dtype=torch.long),
        }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {
        "accuracy" : accuracy_score(labels, preds),
        "f1"       : f1_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall"   : recall_score(labels, preds),
        "roc_auc"  : roc_auc_score(labels, probs)
    }

print("Loading RoBERTa tokenizer...")
tokenizer     = RobertaTokenizer.from_pretrained("roberta-base")
train_dataset = IMDBDataset(train_df, tokenizer)
test_dataset  = IMDBDataset(test_df,  tokenizer)
print(f"✓ Train: {len(train_dataset)} — Test: {len(test_dataset)}")

Loading RoBERTa tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

✓ Train: 25000 — Test: 25000


In [ ]:
def train_roberta_with_rank(r, lora_alpha=None):
    """
    Train RoBERTa + LoRA with a specific rank r.
    lora_alpha defaults to 2*r if not specified.
    """
    if lora_alpha is None:
        lora_alpha = r * 2

    print(f"\n{'='*50}")
    print(f"Training RoBERTa + LoRA with r={r}, alpha={lora_alpha}")
    print(f"{'='*50}")

    base_model = RobertaForSequenceClassification.from_pretrained(
        "roberta-base", num_labels=2
    )

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=r,
        lora_alpha=lora_alpha,
        lora_dropout=0.1,
        target_modules=["query", "value"],
        bias="none"
    )

    model = get_peft_model(base_model, lora_config)
    model = model.to(device)
    model.print_trainable_parameters()

    training_args = TrainingArguments(
        output_dir=f"{DIRS['ablation']}/roberta_r{r}",
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-4,
        warmup_steps=200,
        eval_strategy="epoch",
        save_strategy="no",
        fp16=True,
        seed=SEED,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
    )

    start = time.time()
    trainer.train()
    train_time = time.time() - start

    # Get final metrics
    metrics = trainer.evaluate()
    results = {
        "r"                  : r,
        "lora_alpha"         : lora_alpha,
        "trainable_params"   : sum(p.numel() for p in model.parameters() if p.requires_grad),
        "total_params"       : sum(p.numel() for p in model.parameters()),
        "accuracy"           : round(metrics["eval_accuracy"], 4),
        "f1"                 : round(metrics["eval_f1"], 4),
        "precision"          : round(metrics["eval_precision"], 4),
        "recall"             : round(metrics["eval_recall"], 4),
        "roc_auc"            : round(metrics["eval_roc_auc"], 4),
        "train_time_minutes" : round(train_time / 60, 1),
    }

    print(f"\n✓ r={r} complete:")
    print(f"  Accuracy : {results['accuracy']}")
    print(f"  F1       : {results['f1']}")
    print(f"  Params   : {results['trainable_params']:,} ({results['trainable_params']/results['total_params']*100:.2f}%)")
    print(f"  Time     : {results['train_time_minutes']} minutes")

    del model, trainer, base_model
    torch.cuda.empty_cache()

    return results

print("✓ Training function ready.")

✓ Training function ready.


In [ ]:
ablation_results = []

# r=8
results_r8 = train_roberta_with_rank(r=8)
ablation_results.append(results_r8)
save_results({"r8": results_r8}, "ablation_r8.json")
print("✓ r=8 saved.")

# r=16 (already have this, just add manually)
results_r16 = {
    "r"                  : 16,
    "lora_alpha"         : 32,
    "trainable_params"   : 1181954,
    "total_params"       : 125829124,
    "accuracy"           : 0.9560,
    "f1"                 : 0.9562,
    "precision"          : 0.9523,
    "recall"             : 0.9601,
    "roc_auc"            : 0.9900,
    "train_time_minutes" : 42.2,
}
ablation_results.append(results_r16)
print("✓ r=16 added from existing results.")

# r=32
results_r32 = train_roberta_with_rank(r=32)
ablation_results.append(results_r32)
save_results({"r32": results_r32}, "ablation_r32.json")
print("✓ r=32 saved.")


Training RoBERTa + LoRA with r=8, alpha=16


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,771,778 || all params: 126,418,948 || trainable%: 1.4015


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Roc Auc
1,0.173164,0.146882,0.946840,0.946114,0.959221,0.933360,0.987813
2,0.148630,0.135032,0.954440,0.954405,0.955132,0.953680,0.989653
3,0.120319,0.144164,0.955880,0.956211,0.949090,0.963440,0.989942


Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall,Roc Auc
0.120319,0.144164,3,0.955880,0.956211,0.949090,0.963440,0.989942



✓ r=32 complete:
  Accuracy : 0.9559
  F1       : 0.9562
  Params   : 1,771,778 (1.40%)
  Time     : 24.8 minutes
✓ Results saved: /content/drive/MyDrive/imdb_peft_project/results/ablation_r32.json
✓ r=32 saved.
